# 437. 路径总和 III

给定一个二叉树的根节点 root ，和一个整数 targetSum ，求该二叉树里节点值之和等于 targetSum 的 路径 的数目。

路径 不需要从根节点开始，也不需要在叶子节点结束，但是路径方向必须是向下的（只能从父节点到子节点）。

## 解题思路

由于路径不需要从根节点开始，也不需要在叶子节点结束，常规的自顶向下单一递归无法直接解决。

这道题通常有两种解法：一种是直观但效率略低的双重递归，另一种是优雅且高效的前缀和 + 回溯。

### 双重递归

1. 外层 DFS： 遍历树上的每一个节点，把每一个节点都当作“根节点”。
2. 内层 DFS： 从这个“根节点”出发，向下寻找和为 targetSum 的路径。

In [ ]:
class Solution:
    def pathSum(self, root: Optional[TreeNode], targetSum: int) -> int:
        if not root:
            return 0
            
        # 以当前节点为起点的路径数
        def root_sum(node, target):
            if not node:
                return 0
            count = 0
            if node.val == target:
                count += 1
            # 即使找到了，也要继续向下寻找，因为存在负数，后续加和可能又回到 target
            count += root_sum(node.left, target - node.val)
            count += root_sum(node.right, target - node.val)
            return count
            
        # 当前节点的路径数 + 左子树里的路径数 + 右子树里的路径数
        return root_sum(root, targetSum) + \
               self.pathSum(root.left, targetSum) + \
               self.pathSum(root.right, targetSum)

### 前缀和+回溯

在树的遍历过程中，我们可以记录从根节点到当前节点的前缀和（Prefix Sum）。

如果当前节点的前缀和是 curr_sum，我们要找的路径和是 targetSum，那么我们只需要查找在当前路径上方，是否出现过前缀和为 curr_sum - targetSum 的节点。如果出现过，说明从那个节点到当前节点的路径之和正好等于 targetSum。

核心步骤：
1. 哈希表记录：用一个字典 prefix_map 记录前缀和及其出现的次数。初始化 prefix_map = {0: 1}，表示前缀和为 0 的情况出现了一次（为了兼容从根节点直接开始刚好等于 targetSum 的情况）。
2. 深度优先遍历（DFS）：向下遍历节点，累加当前路径的和 curr_sum。
3. 查表累加：在字典中查找 curr_sum - targetSum 出现的次数，将其加入到总路径数中。
4. 更新状态：将当前的 curr_sum 记录到字典中，继续递归左右子树。
5. 回溯（非常关键）：由于题目要求路径必须是向下的，不能跨越兄弟节点，所以当我们处理完当前节点，准备向上返回父节点时，必须在字典中将当前节点的 curr_sum 次数减 1，撤销它的影响。

In [ ]:
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def pathSum(self, root: Optional[TreeNode], targetSum: int) -> int:
        # 哈希表存储前缀和出现的次数
        # 初始化 {0: 1} 是为了处理从根节点开始的路径正好等于 targetSum 的情况
        prefix_map = {0: 1}
        
        # 在以当前 node 为根节点的这棵子树中，
        # 所有满足“路径和等于 targetSum” 的有效路径的总条数。
        def dfs(node, curr_sum):
            if not node:
                return 0
            
            # 1. 计算当前前缀和
            curr_sum += node.val
            
            # 2. 查找是否有满足条件的前缀和，如果有，累加其出现的次数
            # 目标前缀和 = 当前前缀和 - 目标和
            target = curr_sum - targetSum
            count = prefix_map.get(target, 0)
            
            # 3. 将当前前缀和加入哈希表，供后续的子节点使用
            prefix_map[curr_sum] = prefix_map.get(curr_sum, 0) + 1
            
            # 4. 递归遍历左右子树
            count += dfs(node.left, curr_sum)
            count += dfs(node.right, curr_sum)
            
            # 5. 回溯（状态恢复）
            # 当该节点的左右子树都遍历完，返回上一层时，需要撤销该节点的前缀和
            # 确保它不会影响到其他分支（比如它的兄弟节点所在的另一侧子树）
            prefix_map[curr_sum] -= 1
            
            return count

        return dfs(root, 0)